In [ ]:
# Cell 0 — Kaggle pip installs, path constants, feature flags.
# Runs top-to-bottom. Non-Kaggle (local Windows) path short-circuits installs.

import os, sys, subprocess
from pathlib import Path

IS_KAGGLE = os.path.exists("/kaggle")

if IS_KAGGLE:
    # Pinned, fail-loud installs. bm25s + CharSplit + nltk are small; faiss-gpu is best-effort.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "bm25s>=0.2.0", "CharSplit", "nltk"])
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "faiss-gpu-cu12>=1.14"])
        USE_FAISS_GPU = True
    except subprocess.CalledProcessError:
        print("faiss-gpu-cu12 install failed; falling back to faiss-cpu", flush=True)
        USE_FAISS_GPU = False
    import nltk
    nltk.download("stopwords", quiet=True)
else:
    USE_FAISS_GPU = False  # local Windows dev always CPU faiss

# === Path constants (extends solution.py:20-35 with BGE_M3_DIR + OPUS_MT_DIR) ===
if IS_KAGGLE:
    DATA_DIR    = Path("/kaggle/input/llm-agentic-legal-information-retrieval")
    BGE_M3_DIR  = Path("/kaggle/input/bge-m3")
    OPUS_MT_DIR = Path("/kaggle/input/opus-mt-tc-big-en-de")
    RERANK_DIR  = Path("/kaggle/input/mmarco-reranker")
    OUT_PATH    = Path("/kaggle/working/submission.csv")
else:
    _ROOT = Path(r"C:\Users\Dharun prasanth\OneDrive\Documents\Projects\LLm_Agentic")
    DATA_DIR    = _ROOT / "Data"
    BGE_M3_DIR  = _ROOT / "models" / "bge-m3"
    OPUS_MT_DIR = _ROOT / "models" / "opus-mt-tc-big-en-de"
    RERANK_DIR  = _ROOT / "models" / "mmarco-reranker"
    OUT_PATH    = _ROOT / "submission.csv"

# === Feature flags (INC-02) ===
USE_RERANKER           = True    # Phase 1: on
USE_COURT_CORPUS       = False   # Phase 2+ enables
USE_ENTITY_GRAPH       = False   # Phase 3+ enables
USE_COURT_DENSE_RERANK = False   # Phase 4+ enables
USE_JINA_CROSS_ENCODER = False   # Phase 4+ replaces mmarco
USE_LLM_AUGMENTATION   = False   # Phase 5+ enables
SMOKE                  = os.environ.get("SMOKE", "0") == "1"

# === Hyperparameters (tune on val only) ===
BM25_LAWS_K   = 100
DENSE_LAWS_K  = 100
RRF_K_CONST   = 60
RERANK_K      = 150
SMOKE_LAWS_N  = 5000
SMOKE_VAL_N   = 3
SEED          = 42

print(f"IS_KAGGLE={IS_KAGGLE}  USE_FAISS_GPU={USE_FAISS_GPU}  SMOKE={SMOKE}")


In [ ]:
# Cell 1 — HARD GATE: abort if GPU is not present. AH-1 prevention.
import torch, time, gc, re, random
import numpy as np
import pandas as pd

assert torch.cuda.is_available(), (
    "GPU unavailable. Check Kaggle Settings -> Accelerator -> GPU T4 x2. "
    "AH-1 prevention: this notebook refuses to run on CPU."
)

DEVICE = torch.device("cuda")
_dev_name = torch.cuda.get_device_name(0)
_vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
_vram_free  = torch.cuda.mem_get_info()[0] / 1e9
print(f"Device: {_dev_name}")
print(f"VRAM total: {_vram_total:.1f} GB")
print(f"VRAM free:  {_vram_free:.1f} GB")

# Deterministic seeds
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Small helper reused by every model stage (D-06)
def unload(*objs):
    """Free GPU memory: del refs, gc collect, empty_cache, log VRAM free."""
    for _o in objs:
        del _o
    gc.collect()
    torch.cuda.empty_cache()
    _free, _total = torch.cuda.mem_get_info()
    print(f"  unloaded -> VRAM free: {_free/1e9:.1f}/{_total/1e9:.1f} GB", flush=True)


In [ ]:
# ── Load data ──
t0 = time.time()
laws  = pd.read_csv(DATA_DIR / 'laws_de.csv')
print(f'laws loaded: {len(laws):,} rows [{time.time()-t0:.1f}s]')

t0 = time.time()
train = pd.read_csv(DATA_DIR / 'train.csv')
val   = pd.read_csv(DATA_DIR / 'val.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')
print(f'train: {len(train)}  val: {len(val)}  test: {len(test)} [{time.time()-t0:.1f}s]')

laws_cits = laws['citation'].astype(str).values

# Train frequency counter for fallback
train_cit_counter = Counter()
for _, r in train.iterrows():
    for c in str(r['gold_citations']).split(';'):
        c = c.strip()
        if c: train_cit_counter[c] += 1
print(f'Train unique citations: {len(train_cit_counter):,}')
TOP_TRAIN = [c for c, _ in train_cit_counter.most_common(25)]

In [ ]:
# ── Load E5-large and encode laws corpus ──
print('Loading multilingual-e5-large...')
tokz = AutoTokenizer.from_pretrained(str(E5_DIR))
mdl  = AutoModel.from_pretrained(str(E5_DIR)).to(DEVICE).eval()
# Note: no .half() — keep FP32 for compatibility

def mean_pool(h, mask):
    m = mask.unsqueeze(-1).float()
    return (h * m).sum(1) / m.sum(1).clamp(min=1e-9)

@torch.no_grad()
def encode(texts, batch_size=64, prefix='passage: ', max_len=256):
    out = []
    for i in range(0, len(texts), batch_size):
        batch = [prefix + str(t)[:800] for t in texts[i:i+batch_size]]
        enc = tokz(batch, return_tensors='pt', padding=True,
                   truncation=True, max_length=max_len).to(DEVICE)
        h = mdl(**enc).last_hidden_state
        emb = mean_pool(h, enc['attention_mask']).float()
        emb = torch.nn.functional.normalize(emb, p=2, dim=1)
        out.append(emb.cpu().numpy())
        if i and i % (batch_size * 100) == 0:
            print(f'  encoded {i}/{len(texts)}  [{time.time()-t0:.0f}s elapsed]')
    return np.vstack(out).astype('float32')

print('Encoding laws corpus...')
t0 = time.time()
# Use citation + first 200 chars of text (shorter = faster + less noise)
laws_snippets = [f"{c} {str(t)[:300]}" for c, t in zip(laws['citation'], laws['text'])]
laws_embs = encode(laws_snippets, batch_size=64, max_len=192)
print(f'  shape: {laws_embs.shape} [{time.time()-t0:.1f}s]')
del laws_snippets; gc.collect()

In [ ]:
# ── Extract legal codes from citation strings ──
# For each law citation, get its 'code' (e.g. 'StPO', 'ZGB')
CODE_PAT = re.compile(r'\b([A-Z][A-Za-z]{1,10})\b\s*$')

def citation_code(cit):
    m = CODE_PAT.search(cit)
    return m.group(1) if m else None

# Precompute code for each law citation (for fast boost later)
laws_codes = np.array([citation_code(c) or '' for c in laws_cits])
print(f'Laws with detected code: {(laws_codes != "").sum()}/{len(laws_cits)}')
print(f'Distinct codes: {len(set(laws_codes))}')
top_codes = Counter(laws_codes).most_common(15)
print(f'Top 15 codes: {top_codes}')

# Query code extraction
QUERY_CODES = {'ZGB','OR','StGB','StPO','ZPO','BGG','SchKG','AHVG','IVG','ATSG',
                'BV','BVG','USG','UVPV','IPRG','SVG','UVG','LAI','DBG','MWStG',
                'URG','UWG','AIG','VStG','VStrR','AHVV','VwVG','RTVG','StBOG',
                'KVG','VZAE','ZStV','AuG','LugÜ','NAG','IRSG','PublG'}
CODE_REGEX = re.compile(r'\b(' + '|'.join(QUERY_CODES) + r')\b')

def extract_query_codes(text):
    return set(CODE_REGEX.findall(text))

# Test on val_001
print(f"\nval_001 extracted codes: {extract_query_codes(val.iloc[0]['query'])}")

In [ ]:
# ── Retrieval function ──
def retrieve(q_en, top_k=500):
    q_emb = encode([q_en], batch_size=1, prefix='query: ', max_len=512)[0]
    scores = laws_embs @ q_emb

    # Boost docs whose code matches query-extracted codes
    query_codes = extract_query_codes(q_en)
    if query_codes:
        code_mask = np.isin(laws_codes, list(query_codes))
        scores[code_mask] += 0.1  # significant boost

    top = np.argsort(-scores)[:top_k]
    return [(laws_cits[i], scores[i]) for i in top]

def predict(q_en, top_k):
    cands = retrieve(q_en, top_k=top_k * 3)
    preds = []
    seen = set()
    # First pass: top scoring
    for cit, score in cands:
        if cit not in seen:
            seen.add(cit)
            preds.append(cit)
            if len(preds) >= top_k:
                break
    # Fallback: add train-frequent citations not already in preds
    for cit in TOP_TRAIN:
        if cit not in seen:
            seen.add(cit)
            preds.append(cit)
            if len(preds) >= top_k + 10:
                break
    return preds[:top_k]

In [ ]:
# ── Validate on val ──
def macro_f1(gold_sets, pred_sets):
    f1s = []
    for g, p in zip(gold_sets, pred_sets):
        if not g and not p: f1s.append(1.0); continue
        if not g or not p:  f1s.append(0.0); continue
        tp = len(g & p)
        pr = tp / len(p)
        rc = tp / len(g)
        f1s.append(2 * pr * rc / (pr + rc) if pr + rc else 0.0)
    return float(np.mean(f1s))

print('Processing val...')
val_ranked = []
for i, row in val.iterrows():
    preds = predict(row['query'], top_k=100)
    gold = set(str(row['gold_citations']).split(';'))
    val_ranked.append({'gold': gold, 'ranked': preds})
    tp20 = len(gold & set(preds[:20]))
    tp50 = len(gold & set(preds[:50]))
    print(f'  {row["query_id"]}: gold={len(gold)}  tp@20={tp20}  tp@50={tp50}')

best_f1, best_k = 0, 0
for k in range(1, 101):
    f1 = macro_f1([r['gold'] for r in val_ranked],
                  [set(r['ranked'][:k]) for r in val_ranked])
    if f1 > best_f1: best_f1, best_k = f1, k
print(f'\n*** Val macro-F1 = {best_f1:.4f} @ top-{best_k} ***')

In [ ]:
# ── Generate test predictions ──
print('Processing test...')
rows = []
for i, row in test.iterrows():
    preds = predict(row['query'], top_k=best_k)
    rows.append({'query_id': row['query_id'], 'predicted_citations': ';'.join(preds)})

submission = pd.DataFrame(rows)
submission.to_csv('submission.csv', index=False)
print(f'Saved submission.csv  ({len(submission)} rows)')
submission.head()